# Aegis-AI Training Notebook

This Colab notebook trains the initial EfficientNet-B1 image encoder, extracts embeddings, trains the sklearn classification head, computes the Fisher matrix for EWC, exports ONNX, and saves the replay buffer.

Important: MalImg already contains grayscale malware images. You are not training on live malware binaries in Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q timm torchcam mlflow pefile onnx onnxruntime onnxruntime-tools scikit-learn seaborn joblib

In [ ]:
from pathlib import Path
import json
import os
import pickle
from collections import defaultdict

import joblib
import mlflow
import numpy as np
import seaborn as sns
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import matplotlib.pyplot as plt

print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
CLASS_LABELS = [
    'Adialer.C','Agent.FYI','Allaple.A','Allaple.L','Alueron.gen!J','Autorun.K',
    'C2LOP.gen!g','C2LOP.P','Dialplatform.B','Dontovo.A','Fakerean','Instantaccess',
    'Lolyda.AA1','Lolyda.AA2','Lolyda.AA3','Lolyda.AT','Malex.gen!J','Obfuscator.AD',
    'Rbot!gen','Skintrim.N','Swizzor.gen!E','Swizzor.gen!I','VB.AT','Wintrim.BX','Yuner.A','Benign'
]
DATASET_ROOT = Path('/content/drive/MyDrive/AegisAI/malimg')
OUTPUT_ROOT = Path('/content/drive/MyDrive/AegisAI/outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MLFLOW_URI = f'file:{OUTPUT_ROOT / "mlruns"}'
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('aegis-ai-malimg')

In [ ]:
class MalImgDataset(Dataset):
    def __init__(self, records, transform):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        path, label = self.records[idx]
        image = Image.open(path).convert('RGB')
        return self.transform(image), CLASS_LABELS.index(label)

records = []
for family_dir in sorted(DATASET_ROOT.iterdir()):
    if family_dir.is_dir() and family_dir.name in CLASS_LABELS:
        for image_path in family_dir.glob('*.png'):
            records.append((image_path, family_dir.name))

train_records, temp_records = train_test_split(records, test_size=0.2, random_state=42, stratify=[label for _, label in records])
val_records, test_records = train_test_split(temp_records, test_size=0.5, random_state=42, stratify=[label for _, label in temp_records])

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_loader = DataLoader(MalImgDataset(train_records, transform), batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(MalImgDataset(val_records, transform), batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(MalImgDataset(test_records, transform), batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
len(train_records), len(val_records), len(test_records)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = timm.create_model('efficientnet_b1', pretrained=True, num_classes=len(CLASS_LABELS)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler()
best_val_acc = 0.0
checkpoint_path = OUTPUT_ROOT / 'efficientnet_b1_best.pth'

In [ ]:
def evaluate(loader):
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
            truths.extend(labels.cpu().numpy().tolist())
    return accuracy_score(truths, preds), preds, truths

with mlflow.start_run(run_name='initial_training'):
    for epoch in range(20):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += float(loss.item())

        val_acc, _, _ = evaluate(val_loader)
        avg_loss = running_loss / max(len(train_loader), 1)
        mlflow.log_metric('train_loss', avg_loss, step=epoch + 1)
        mlflow.log_metric('val_accuracy', val_acc, step=epoch + 1)
        torch.save({'epoch': epoch + 1, 'state_dict': model.state_dict()}, OUTPUT_ROOT / f'checkpoint_epoch_{epoch+1}.pth')
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), checkpoint_path)
        print(f'Epoch {epoch+1}/20 - loss: {avg_loss:.4f} - val_acc: {val_acc:.4f}')

    mlflow.log_metric('best_val_accuracy', best_val_acc)

In [ ]:
model.load_state_dict(torch.load(checkpoint_path, map_location=device))

feature_extractor = timm.create_model('efficientnet_b1', pretrained=False, num_classes=0, global_pool='avg').to(device)
state_dict = model.state_dict()
feature_extractor.load_state_dict({k: v for k, v in state_dict.items() if k in feature_extractor.state_dict()}, strict=False)
feature_extractor.eval()

def collect_embeddings(loader):
    embeddings, labels = [], []
    with torch.no_grad():
        for images, y in loader:
            images = images.to(device)
            emb = feature_extractor(images)
            embeddings.append(emb.cpu().numpy())
            labels.append(y.numpy())
    return np.concatenate(embeddings), np.concatenate(labels)

train_emb, train_y = collect_embeddings(train_loader)
test_emb, test_y = collect_embeddings(test_loader)
head = LogisticRegression(max_iter=2000, multi_class='multinomial')
head.fit(train_emb, train_y)
test_pred = head.predict(test_emb)
print('Head accuracy:', accuracy_score(test_y, test_pred))
joblib.dump(head, OUTPUT_ROOT / 'head.pkl')

In [ ]:
cm = confusion_matrix(test_y, test_pred)
plt.figure(figsize=(14, 10))
sns.heatmap(cm, cmap='mako')
plt.title('MalImg Test Confusion Matrix')
plt.show()
print(classification_report(test_y, test_pred, target_names=CLASS_LABELS[:len(set(test_y))]))

In [ ]:
fisher = {name: torch.zeros_like(param) for name, param in model.named_parameters()}
model.eval()
for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)
    model.zero_grad(set_to_none=True)
    loss = criterion(model(images), labels)
    loss.backward()
    for name, param in model.named_parameters():
        if param.grad is not None:
            fisher[name] += param.grad.detach().cpu().pow(2)

for name in fisher:
    fisher[name] /= max(len(train_loader), 1)

with open(OUTPUT_ROOT / 'fisher_matrix.pkl', 'wb') as handle:
    pickle.dump(fisher, handle)

In [ ]:
replay_buffer = []
per_family = defaultdict(int)
for image_path, label in train_records:
    if per_family[label] >= 8:
        continue
    image = Image.open(image_path).convert('RGB').resize((224, 224))
    array = np.asarray(image).astype(np.float32) / 255.0
    array = np.transpose(array, (2, 0, 1))
    replay_buffer.append({'image': array.tolist(), 'label': label, 'label_index': CLASS_LABELS.index(label), 'path': str(image_path)})
    per_family[label] += 1
    if sum(per_family.values()) >= 200:
        break

with open(OUTPUT_ROOT / 'replay_buffer.pkl', 'wb') as handle:
    pickle.dump(replay_buffer, handle)

validation_set = []
validation_counts = defaultdict(int)
for image_path, label in val_records + test_records:
    if validation_counts[label] >= 100:
        continue
    image = Image.open(image_path).convert('RGB').resize((224, 224))
    array = np.asarray(image).astype(np.float32) / 255.0
    array = np.transpose(array, (2, 0, 1))
    validation_set.append({'image': array.tolist(), 'label': label, 'label_index': CLASS_LABELS.index(label), 'path': str(image_path)})
    validation_counts[label] += 1

with open(OUTPUT_ROOT / 'validation_set.pkl', 'wb') as handle:
    pickle.dump(validation_set, handle)

In [ ]:
class EmbeddingWrapper(nn.Module):
    def __init__(self, inner_model):
        super().__init__()
        self.inner_model = inner_model

    def forward(self, x):
        features = self.inner_model.forward_features(x)
        pooled = self.inner_model.global_pool(features)
        if pooled.ndim > 2:
            pooled = torch.flatten(pooled, 1)
        return pooled

model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()
dummy = torch.randn(1, 3, 224, 224, device=device)
onnx_path = OUTPUT_ROOT / 'efficientnet_b1.onnx'
embedding_model = EmbeddingWrapper(model).to(device)
torch.onnx.export(embedding_model, dummy, str(onnx_path), input_names=['input'], output_names=['embedding'], opset_version=17)
torch.save(model.state_dict(), OUTPUT_ROOT / 'efficientnet_b1_pytorch.pth')
print('Saved:', onnx_path)

After the notebook finishes, download these files from `MyDrive/AegisAI/outputs/` and place them in your local project:

- `efficientnet_b1.onnx`
- `efficientnet_b1_pytorch.pth`
- `head.pkl`
- `fisher_matrix.pkl`
- `replay_buffer.pkl`
- `validation_set.pkl`